# Dimension Products Model
**Sources:** crm_products, erp_products  
**Target:** dim_products  

**Columns to include:**  
--- Columns in crm_products ---  
product_id  
product_name = True    
product_cost   
product_line = True    
start_date  
end_date  
end_date_test  
category_id = True   
product_key = True   
silver_updated_at  

--- Columns in erp_products ---  
category = True  
sub_category = True  
maintenance = True  
category_id = True  
silver_updated_at  

# Explore Tables

In [0]:
%sql
USE CATALOG databricks_bootcamp_dwb;
USE SCHEMA silver;

In [0]:
tables = ["crm_products", "erp_products"]

for t in tables:
    print(f"--- {t} ---")
    display(spark.table(t).limit(5))
    print()

In [0]:
# Validate that each product_key has at most 1 non-null value for each included column
from pyspark.sql import functions as F

# Columns to validate (marked as = True)
cols_to_check = ['product_name', 'product_line', 'category_id', 'category', 'sub_category', 'maintenance']

# Join tables and check for multiple distinct non-null values per product_key
validation = spark.sql("""
    SELECT 
        p.product_key,
        COUNT(DISTINCT p.product_name) as distinct_product_names,
        COUNT(DISTINCT p.product_line) as distinct_product_lines,
        COUNT(DISTINCT p.category_id) as distinct_category_ids,
        COUNT(DISTINCT e.category) as distinct_categories,
        COUNT(DISTINCT e.sub_category) as distinct_sub_categories,
        COUNT(DISTINCT e.maintenance) as distinct_maintenance
    FROM databricks_bootcamp_dwb.silver.crm_products p
    LEFT JOIN databricks_bootcamp_dwb.silver.erp_products e ON p.category_id = e.category_id
    GROUP BY p.product_key
    HAVING 
        distinct_product_names > 1 OR
        distinct_product_lines > 1 OR
        distinct_category_ids > 1 OR
        distinct_categories > 1 OR
        distinct_sub_categories > 1 OR
        distinct_maintenance > 1
""")

violation_count = validation.count()
print(f"Product keys with multiple distinct values: {violation_count}")

if violation_count > 0:
    print("\n⚠️ Data quality issue detected!")
    display(validation)
else:
    print("✅ All product_keys have consistent single values")

# Perform Business Transformations and Modeling

In [0]:
# Join the two source tables and select required columns (only columns with = True)
# Drop duplicates, keeping the first instance of each product_key
df = spark.sql("""
    SELECT 
        p.product_key,
        p.category_id,
        e.category,
        e.sub_category,
        p.product_line,
        p.product_name,
        e.maintenance
    FROM crm_products p
    LEFT JOIN erp_products e ON p.category_id = e.category_id
""").dropDuplicates(["product_key"])

# Write Gold Table

In [0]:
# Write to gold layer
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("databricks_bootcamp_dwb.gold.dim_products")

In [0]:
%sql
SELECT *
FROM databricks_bootcamp_dwb.gold.dim_products
LIMIT 100;